# RumiFold demo

Fold a protein and ask about it. The ESM-C embeddings are bundled with this repository, so this needs only Qwen3-8B and one RumiFold checkpoint. A 48 GB GPU is comfortable.

In [ ]:
import json, sys
sys.path.insert(0, '..')
from rumifold import RumiFold, load_features, write_pdb

name = 'lumazine_synthase'   # or 'lysozyme'
info = json.load(open('data/proteins.json'))[name]
print(info['uniprot'], info['length'], 'residues')
print(info['sequence'])

In [ ]:
# downloads the checkpoint and Qwen3-8B on first use
model = RumiFold.from_pretrained('rumifold-156k')

## Fold

In [ ]:
fold = model.fold(load_features(f'data/{name}.safetensors'))
write_pdb('demo.pdb', fold.backbone, info['sequence'])
fold.backbone.shape   # residues, (N, CA, C), xyz

In [ ]:
# optional: pip install py3Dmol
try:
    import py3Dmol
    v = py3Dmol.view(width=500, height=400); v.addModel(open('demo.pdb').read(), 'pdb')
    v.setStyle({'cartoon': {'color': 'spectrum'}}); v.zoomTo(); v.show()
except ImportError:
    print('install py3Dmol to view the structure here, or open demo.pdb in PyMOL')

## Ask

In [ ]:
questions = ['Which residues form the active site of this protein?',
    "First list this protein's Gene Ontology molecular function terms (most specific first) and EC number if any, "
    'then describe its function in one or two sentences, as a Swiss-Prot FUNCTION entry would.']
for q in questions:
    print('Q:', q); print('A:', model.ask(q, fold)); print()

It is still a language model, so it can also answer without a protein:

In [ ]:
print(model.ask('What is the capital of France?'))

## Your own protein

Two ways to get embeddings: compute them with ESM-C 6B (`scripts/make_esmc_features.py`), or read ours by byte range if the exact sequence is in a RumiFold dataset.

In [ ]:
from rumifold import features_from_hub
x = features_from_hub(info['sequence'], 'RumiFold/RumiFold_Functional')   # RumiFold/RumiFold_GO by default
x.shape